In [1]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
from scipy.stats import qmc

### Function Description

Address the challenge of optimally placing products across warehouses for a business with high online sales, where accurate calculations are costly and only feasible biweekly. To speed up decision-making, an ML model approximates these results within hours. The model has four hyperparameters to tune, and its output reflects the difference from the expensive baseline. Because the system is dynamic and full of local optima, it requires careful tuning and robust validation to find reliable, near-optimal solutions.


In [2]:
# Load data
X = np.load('initial_inputs.npy')
Y = np.load('initial_outputs.npy').ravel()

In [3]:
df = pd.DataFrame(X, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y
print(df.describe())

              x1         x2         x3         x4     target
count  30.000000  30.000000  30.000000  30.000000  30.000000
mean    0.542872   0.477129   0.465693   0.474096 -17.238587
std     0.294007   0.308910   0.261618   0.292412   7.137959
min     0.037825   0.006250   0.042186   0.081517 -32.625660
25%     0.258744   0.150459   0.223273   0.215885 -21.578590
50%     0.601918   0.499909   0.445999   0.499451 -16.040082
75%     0.787653   0.749126   0.708686   0.758504 -12.745906
max     0.985622   0.919592   0.939178   0.999483  -4.025542


In [4]:
# Surrogate model setup
initial_length_scales = np.array([0.5, 0.5, 0.5, 0.5])
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 2.0), nu=2.5) + \
         WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-5, 1e-1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    normalize_y=True, 
    n_restarts_optimizer=25,
    random_state=42
)

# Fit the model
model.fit(X, Y)

# 4D RANDOM SEARCH GRID
np.random.seed(42)
grid_size = 100000 
search_grid = np.random.uniform(0, 1, size=(grid_size, 4))

# PREDICT
mean, std = model.predict(search_grid, return_std=True)

# BETA SWEEP FOR ACQUISITION
betas = [0.1, 0.5, 1.0, 1.5, 2.0, 3.0]
sweep_results = []

for b in betas:
    # Upper Confidence Bound calculation
    ucb_scores = mean + (b * std)
    
    # Get index of the highest score
    best_idx = np.argmax(ucb_scores)
    best_point = search_grid[best_idx]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_point[0],
        'x2': best_point[1],
        'x3': best_point[2],
        'x4': best_point[3],
        'Pred Mean': mean[best_idx],
        'Uncertainty': std[best_idx]
    })

# DISPLAY RESULTS
df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n--- Optimized Model Parameters ---")
print(model.kernel_)

--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---
  Beta     x1     x2     x3     x4  Pred Mean  Uncertainty
0.1000 0.4542 0.4743 0.4113 0.3970    -1.8595       1.3893
0.5000 0.4425 0.4926 0.3669 0.4543    -1.9694       1.6351
1.0000 0.4425 0.4926 0.3669 0.4543    -1.9694       1.6351
1.5000 0.4425 0.4926 0.3669 0.4543    -1.9694       1.6351
2.0000 0.4146 0.4786 0.3580 0.4746    -2.2110       1.7609
3.0000 0.4146 0.4786 0.3580 0.4746    -2.2110       1.7609

--- Optimized Model Parameters ---
Matern(length_scale=[0.667, 0.538, 0.713, 0.632], nu=2.5) + WhiteKernel(noise_level=1e-05)


/opt/homebrew/anaconda3/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


### Week 1 submission: x1 = 0.4425, x2 = 0.4926, x3 = 0.3669, x4=0.4543

* The Problem Framing: We are tuning a machine learning model with 4 hyperparameters. The target value is the negative difference from an expensive baseline calculation.

* The Surrogate Model: Matern 2.5 + ARD : We initially considered Rational Quadratic, but switched to Matern 2.5 for rock-solid architectural stability in 4D space. Matern perfectly balances smoothness (broad trends) with sharpness (local peaks).
    * ARD (Automatic Relevance Determination): By giving each of the 4 dimensions its own length scale, the model learns which knobs act as "fine-tuning". 

* Engineering Fixes:
    * normalize_y=True: Essential because the targets are entirely negative. It stops the model from assuming unexplored 4D space defaults to an artificially attractive 0.0.
    * Random Search Grid: Replaced the standard geometric grid with a 100,000-point Monte Carlo random sample. This bypasses the "curse of dimensionality" and prevents memory crashes while still providing a highly dense search space.
    * Shape Enforcement: Added strict .ravel() and explicit numpy array definitions to prevent matrix mismatch errors during gradient descent.

* Key Insight from the Model: Hyper-Confidence. The model hit the lower bound for noise (1e-05), meaning it views your current 30 data points as highly deterministic.
    * Because of this lack of assumed noise, the model's Predicted Mean (μ) is dominating the Unceartainty (σ) equation, even at high Beta values.

The Next Query: [0.4425 , 0.4926 , 0.3669 , 0.4543]

* This is a pure "Exploitation" play. The model is overwhelmingly confident that balancing all four parameters tightly in the middle of the hypercube will yield a massive jump in performance (predicting −1.96, which is twice as good as your current best of −4.02).

In [5]:
# ---------------------------------------------------
# WEEK1: NEW DATA HERE
# ---------------------------------------------------
w1_new_input = [0.4425, 0.4926, 0.3669, 0.4543]
w1_new_output = -1.46093545716754

new_X = np.array(w1_new_input).reshape(1, -1)
new_Y = np.atleast_1d(w1_new_output)

X_updated_w1 = np.vstack((X, new_X))
Y_updated_w1 = np.append(Y, new_Y)

df = pd.DataFrame(X_updated_w1, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w1

print(df.describe())
print(df.head(31))


              x1         x2         x3         x4     target
count  31.000000  31.000000  31.000000  31.000000  31.000000
mean    0.539635   0.477628   0.462506   0.473458 -16.729631
std     0.289627   0.303731   0.257832   0.287519   7.568504
min     0.037825   0.006250   0.042186   0.081517 -32.625660
25%     0.266542   0.155668   0.229306   0.225204 -21.048893
50%     0.577766   0.499588   0.438806   0.494932 -16.026400
75%     0.774002   0.741294   0.708006   0.746945 -12.711725
max     0.985622   0.919592   0.939178   0.999483  -1.460935
          x1        x2        x3        x4     target
0   0.896981  0.725628  0.175404  0.701694 -22.108288
1   0.889356  0.499588  0.539269  0.508783 -14.601397
2   0.250946  0.033693  0.145380  0.494932 -11.699932
3   0.346962  0.006250  0.760564  0.613024 -16.053765
4   0.124871  0.129770  0.384400  0.287076 -10.069633
5   0.801303  0.500231  0.706645  0.195103 -15.487083
6   0.247708  0.060445  0.042186  0.441324 -12.681685
7   0.746702  0.757

In [6]:
# Surrogate model setup
initial_length_scales = np.array([0.5, 0.5, 0.5, 0.5])
kernel = Matern(length_scale=initial_length_scales, length_scale_bounds=(0.01, 5.0), nu=2.5) + \
         WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-10, 1e-1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    normalize_y=True, 
    n_restarts_optimizer=25,
    random_state=42
)

# Fit the model
model.fit(X_updated_w1, Y_updated_w1)

# 4D RANDOM SEARCH GRID
np.random.seed(42)
grid_size = 100000 
search_grid = np.random.uniform(0, 1, size=(grid_size, 4))

# PREDICT
mean, std = model.predict(search_grid, return_std=True)

# BETA SWEEP FOR ACQUISITION
betas = [0.1, 0.5, 1.0, 1.5, 2.0, 3.0]
sweep_results = []

for b in betas:
    # Upper Confidence Bound calculation
    ucb_scores = mean + (b * std)
    
    # Get index of the highest score
    best_idx = np.argmax(ucb_scores)
    best_point = search_grid[best_idx]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_point[0],
        'x2': best_point[1],
        'x3': best_point[2],
        'x4': best_point[3],
        'Pred Mean': mean[best_idx],
        'Uncertainty': std[best_idx]
    })

# DISPLAY RESULTS
df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n--- Optimized Model Parameters ---")
print(model.kernel_)

--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---
    Beta       x1       x2       x3       x4  Pred Mean  Uncertainty
0.100000 0.454159 0.474299 0.411280 0.397026  -1.428372     0.376421
0.500000 0.454159 0.474299 0.411280 0.397026  -1.428372     0.376421
1.000000 0.454159 0.474299 0.411280 0.397026  -1.428372     0.376421
1.500000 0.397586 0.438602 0.446588 0.398243  -1.887565     0.779554
2.000000 0.397586 0.438602 0.446588 0.398243  -1.887565     0.779554
3.000000 0.506951 0.454392 0.204707 0.381093  -3.222788     1.393571

--- Optimized Model Parameters ---
Matern(length_scale=[0.729, 0.59, 0.757, 0.685], nu=2.5) + WhiteKernel(noise_level=1.69e-10)


### Week 2 submission:

* Strategy Pivot: We are intentionally stepping away from the previous high-exploration strategy (β=2.0) to hone in on this successful region. This is a shift into Phase 2: Localization, where we attempt to find the exact peak of the local optimum we just discovered.

* The Consensus Signal: The Beta sweep shows a unified front; β=0.1 through β=1.0 all point to the exact same coordinate. This indicates that the "Exploitation" and "Balanced" strategies have converged on a single high-confidence target.

* The Chosen Query: [0.45415862, 0.47429924, 0.41128020, 0.39702632] (Beta = 0.1 - 1)


In [7]:
# ---------------------------------------------------
# WEEK2: NEW DATA HERE
# ---------------------------------------------------

w2_new_input = [0.454159, 0.474299, 0.411280, 0.397026]
w2_new_output = np.float64(-1.38893396432946)

w2_new_X = np.atleast_2d(w2_new_input)
w2_new_Y = np.atleast_1d(w2_new_output)

X_updated_w2 = np.vstack((X_updated_w1, w2_new_X))
Y_updated_w2 = np.append(Y_updated_w1, w2_new_Y)

scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_updated_w2_scaled = scaler_x.fit_transform(X_updated_w2)
Y_updated_w2_scaled = scaler_y.fit_transform(Y_updated_w2.reshape(-1, 1))

df = pd.DataFrame(X_updated_w2, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w2

# print(df.describe())
print(df.head(32))

          x1        x2        x3        x4     target
0   0.896981  0.725628  0.175404  0.701694 -22.108288
1   0.889356  0.499588  0.539269  0.508783 -14.601397
2   0.250946  0.033693  0.145380  0.494932 -11.699932
3   0.346962  0.006250  0.760564  0.613024 -16.053765
4   0.124871  0.129770  0.384400  0.287076 -10.069633
5   0.801303  0.500231  0.706645  0.195103 -15.487083
6   0.247708  0.060445  0.042186  0.441324 -12.681685
7   0.746702  0.757092  0.369353  0.206566 -16.026400
8   0.400665  0.072574  0.886768  0.243842 -17.049235
9   0.626071  0.586751  0.438806  0.778858 -12.741766
10  0.957135  0.597644  0.766114  0.776210 -27.316396
11  0.732812  0.145250  0.476813  0.133366 -13.527649
12  0.655115  0.072392  0.687152  0.081517 -16.679115
13  0.219734  0.832031  0.482864  0.082569 -16.507159
14  0.488594  0.211965  0.939178  0.376192 -17.817999
15  0.167130  0.876555  0.217240  0.959801 -26.561821
16  0.216911  0.166086  0.241372  0.770062 -12.758324
17  0.387488  0.804532  0.75

In [8]:

kernel = Matern(length_scale=[0.5]*4, length_scale_bounds=(0.01, 8.0), nu=2.5) + \
         WhiteKernel(noise_level=1e-3, noise_level_bounds=(1e-12, 1e-1))

model = GaussianProcessRegressor(
    kernel=kernel, 
    n_restarts_optimizer=25,
    random_state=42
)

model.fit(X_updated_w2_scaled, Y_updated_w2_scaled)

# Define the Acquisition Function (UCB) for the Optimizer
def acquisition_function(x_trial, beta, model, scaler_x):
    x_trial = x_trial.reshape(1, -1)
    mean, std = model.predict(x_trial, return_std=True)
    ucb = mean[0] + (beta * std[0])
    return -ucb  

betas = [0.01, 0.05, 0.1 , 0.15, 0.20, 0.25]
sweep_results = []

# Generate a Sobol sequence 
sampler = qmc.Sobol(d=4, scramble=True, seed=42)
sample_raw = sampler.random(n=8192)
sample_scaled = scaler_x.transform(sample_raw)

for b in betas:
    # Find the best starting point from the sample
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    # Use L-BFGS-B to find the absolute peak near that starting point
    bounds = [scaler_x.transform([[0,0,0,0]])[0], scaler_x.transform([[1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model, scaler_x),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)

    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })


# DISPLAY RESULTS
df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

print("\n--- Optimized Model Parameters ---")
print(model.kernel_)

--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---
    Beta       x1       x2       x3       x4  Pred Mean  Uncertainty
0.010000 0.447606 0.472733 0.381083 0.413062  -1.311659     0.108528
0.050000 0.447606 0.471712 0.379817 0.412494  -1.311972     0.118950
0.100000 0.447599 0.470375 0.378233 0.411809  -1.312976     0.132324
0.150000 0.447592 0.468977 0.376662 0.411161  -1.314684     0.145976
0.200000 0.447567 0.467522 0.375109 0.410554  -1.317116     0.159871
0.250000 0.447526 0.466008 0.373583 0.409993  -1.320289     0.173969

--- Optimized Model Parameters ---
Matern(length_scale=[2.77, 2.15, 3.2, 2.63], nu=2.5) + WhiteKernel(noise_level=3.24e-12)


### Week 3 submission:

* 4D Density Trap: Replaced the memory-heavy np.meshgrid with a Hybrid Search Strategy. We now use a Sobol sequence, followed by the L-BFGS-B local optimizer to climb to the exact continuous mathematical peak.

* Feature Standardization: Applied StandardScaler to the X coordinates, shifting the 0-to-1 space into Z-scores. 
    *  Kernel Boundary Expansion: Increased the length_scale_bounds from 2.0 to 8.0. The kernel needed this higher ceiling to accurately model the broad, smooth gradients of your optimum zone.

* Query Strategy: Pure Exploitation (β=0.1) [0.447599 , 0.470376 , 0.378234 , 0.411809] Predicted Target: -1.312976

In [9]:
# ---------------------------------------------------
# WEEK3: NEW DATA HERE
# ---------------------------------------------------

w3_new_input = [0.447599, 0.470376, 0.378234, 0.411809]
w3_new_output = -0.9991309634879637

w3_new_X = np.atleast_2d(w3_new_input)
w3_new_Y = np.atleast_1d(w3_new_output)

X_updated_w3 = np.vstack((X_updated_w2, w3_new_X))
Y_updated_w3 = np.append(Y_updated_w2, w3_new_Y)

X_updated_w3_scaled = scaler_x.fit_transform(X_updated_w3)
Y_updated_w3_scaled = scaler_y.fit_transform(Y_updated_w3.reshape(-1, 1))

df = pd.DataFrame(X_updated_w3, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w3

print(df.describe())


              x1         x2         x3         x4     target
count  33.000000  33.000000  33.000000  33.000000  33.000000
mean    0.534255   0.477307   0.458400   0.469273 -15.788079
std     0.281255   0.294089   0.250218   0.278898   8.238625
min     0.037825   0.006250   0.042186   0.081517 -32.625660
25%     0.282138   0.166086   0.241372   0.243842 -19.989498
50%     0.488594   0.493965   0.425826   0.454300 -15.487083
75%     0.746702   0.725628   0.706645   0.723827 -11.699932
max     0.985622   0.919592   0.939178   0.999483  -0.999131


In [10]:
model.fit(X_updated_w3_scaled, Y_updated_w3_scaled)

betas = [0.01, 0.05, 0.10 , 0.15, 0.20, 0.25]
sweep_results = []

# Generate a Sobol sequence 
sampler = qmc.Sobol(d=4, scramble=True, seed=42)
sample_raw = sampler.random(n=8192)
sample_scaled = scaler_x.transform(sample_raw)

for b in betas:
    # Find the best starting point from the sample
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    # Use L-BFGS-B to find the absolute peak near that starting point
    bounds = [scaler_x.transform([[0,0,0,0]])[0], scaler_x.transform([[1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model, scaler_x),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)

    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })


# DISPLAY RESULTS
df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\n--- Optimized Model Parameters ---")
print(model.kernel_)

--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---
    Beta       x1       x2       x3       x4  Pred Mean  Uncertainty
0.010000 0.446587 0.452347 0.354953 0.403755  -1.074634     0.298622
0.050000 0.446578 0.451589 0.354013 0.403554  -1.074797     0.304036
0.100000 0.446572 0.450613 0.352808 0.403311  -1.075326     0.311069
0.150000 0.446570 0.449605 0.351573 0.403075  -1.076244     0.318402
0.200000 0.446571 0.448567 0.350307 0.402848  -1.077580     0.326030
0.250000 0.446580 0.447499 0.349017 0.402632  -1.079360     0.333934

--- Optimized Model Parameters ---
Matern(length_scale=[2.83, 2.28, 3.37, 2.7], nu=2.5) + WhiteKernel(noise_level=0.000383)


### Week 4 submission: 

Next point to query [0.446587 , 0.452347 , 0.354953 , 0.403755] beta = 0.01 

* Hyperoptimizing by changing the beta from 0.1 to 0.01 
* Based on the observed improvement, if there's a similar jump in "performance" it's worth the conitnuous optmization, if not - or if the value plateaus, next round can consider more explorative approach. 

In [11]:
# ---------------------------------------------------
# WEEK4: NEW DATA HERE
# ---------------------------------------------------

w4_new_input = 	[0.446587, 0.452347, 0.354953, 0.403755]
w4_new_output = -0.23126888515545074

w4_new_X = np.atleast_2d(w4_new_input)
w4_new_Y = np.atleast_1d(w4_new_output)

X_updated_w4 = np.vstack((X_updated_w3, w4_new_X))
Y_updated_w4 = np.append(Y_updated_w3, w4_new_Y)

X_updated_w4_scaled = scaler_x.fit_transform(X_updated_w4)
Y_updated_w4_scaled = scaler_y.fit_transform(Y_updated_w4.reshape(-1, 1))

df = pd.DataFrame(X_updated_w4, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w4

print(df.describe())

              x1         x2         x3         x4     target
count  34.000000  34.000000  34.000000  34.000000  34.000000
mean    0.531677   0.476573   0.455357   0.467346 -15.330526
std     0.277369   0.289630   0.247036   0.274869   8.540269
min     0.037825   0.006250   0.042186   0.081517 -32.625660
25%     0.293123   0.177556   0.250159   0.245134 -19.852513
50%     0.471377   0.493282   0.418553   0.447812 -15.044240
75%     0.744680   0.710954   0.701771   0.718294 -11.599290
max     0.985622   0.919592   0.939178   0.999483  -0.231269


In [12]:
model.fit(X_updated_w4_scaled, Y_updated_w4_scaled)

betas = [0.01, 0.05, 0.10 , 0.15, 0.20]
sweep_results = []

# Generate a Sobol sequence 
sampler = qmc.Sobol(d=4, scramble=True, seed=42)
sample_raw = sampler.random(n=8192)
sample_scaled = scaler_x.transform(sample_raw)

for b in betas:
    # Find the best starting point from the sample
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    # Use L-BFGS-B to find the absolute peak near that starting point
    bounds = [scaler_x.transform([[0,0,0,0]])[0], scaler_x.transform([[1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model, scaler_x),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)

    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })


# DISPLAY RESULTS
df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\n--- Optimized Model Parameters ---")
print(model.kernel_)

--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---
    Beta       x1       x2       x3       x4  Pred Mean  Uncertainty
0.010000 0.448426 0.432928 0.312796 0.400045  -0.529421     0.545977
0.050000 0.448601 0.432485 0.311611 0.399966  -0.529583     0.551337
0.100000 0.448838 0.431935 0.310090 0.399867  -0.530103     0.558258
0.150000 0.449096 0.431391 0.308520 0.399767  -0.531001     0.565435
0.200000 0.449377 0.430855 0.306901 0.399666  -0.532305     0.572877

--- Optimized Model Parameters ---
Matern(length_scale=[2.77, 2.27, 3.5, 2.68], nu=2.5) + WhiteKernel(noise_level=0.00142)


### Week 5 submission:

Next point to query: [0.448601 0.432485 0.311611 0.399966] beta = 0.05

* Improved on the last run, keep optimising 
* Uncertainty strangely has gone up - see if it means anything - check against the result from this round

In [13]:
# ---------------------------------------------------
# WEEK5: NEW DATA HERE
# ---------------------------------------------------

w5_new_input = 	[0.448601, 0.432485, 0.311611, 0.399966]
w5_new_output = -0.7607751429561804

w5_new_X = np.atleast_2d(w5_new_input)
w5_new_Y = np.atleast_1d(w5_new_output)

X_updated_w5 = np.vstack((X_updated_w4, w5_new_X))
Y_updated_w5 = np.append(Y_updated_w4, w5_new_Y)

X_updated_w5_scaled = scaler_x.fit_transform(X_updated_w5)
Y_updated_w5_scaled = scaler_y.fit_transform(Y_updated_w5.reshape(-1, 1))

df = pd.DataFrame(X_updated_w5, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w5

print(df.describe())

              x1         x2         x3         x4     target
count  35.000000  35.000000  35.000000  35.000000  35.000000
mean    0.529303   0.475313   0.451250   0.465421 -14.914248
std     0.273620   0.285437   0.244585   0.271036   8.766760
min     0.037825   0.006250   0.042186   0.081517 -32.625660
25%     0.304107   0.189025   0.258946   0.246425 -19.715528
50%     0.454159   0.492600   0.411280   0.441324 -14.601397
75%     0.742658   0.696280   0.696898   0.712761 -10.817688
max     0.985622   0.919592   0.939178   0.999483  -0.231269


In [14]:
model.fit(X_updated_w5_scaled, Y_updated_w5_scaled)

betas = [0.01, 0.10 , 0.25, 0.5, 1.0, 1.5, 2.5]
sweep_results = []

# Generate a Sobol sequence 
sampler = qmc.Sobol(d=4, scramble=True, seed=42)
sample_raw = sampler.random(n=8192)
sample_scaled = scaler_x.transform(sample_raw)

for b in betas:
    # Find the best starting point from the sample
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    # Use L-BFGS-B to find the absolute peak near that starting point
    bounds = [scaler_x.transform([[0,0,0,0]])[0], scaler_x.transform([[1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model, scaler_x),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)

    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })


# DISPLAY RESULTS
df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\n--- Optimized Model Parameters ---")
print(model.kernel_)

--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---
    Beta       x1       x2       x3       x4  Pred Mean  Uncertainty
0.010000 0.445232 0.436879 0.327133 0.402145  -0.678095     0.383883
0.100000 0.445105 0.436285 0.326495 0.402133  -0.678170     0.385232
0.250000 0.444859 0.435197 0.325417 0.402145  -0.678605     0.387695
0.500000 0.444298 0.432993 0.323681 0.402314  -0.680471     0.392603
1.000000 0.439897 0.423159 0.324689 0.405134  -0.699369     0.415600
1.500000 0.424922 0.400493 0.342756 0.415644  -0.853823     0.536750
2.500000 0.489702 0.483486 0.213283 0.382209  -1.742763     0.965623

--- Optimized Model Parameters ---
Matern(length_scale=[2.94, 2.41, 3.62, 2.83], nu=2.5) + WhiteKernel(noise_level=0.00134)


### Week 6 submission

Next point to query: [0.424922 0.400493 0.342756 0.415644] beta = 1.5

* Switching into exploration mode - looking for another local optima 
* Expect a higher output, but stick to trying to optimize 
* Strategy is to run 2 sequential queries to search for potential 

In [15]:
# ---------------------------------------------------
# WEEK6: NEW DATA HERE
# ---------------------------------------------------

w6_new_input = 	[0.424922, 0.400493, 0.342756, 0.415644]
w6_new_output = -0.15046741058640586

w6_new_X = np.atleast_2d(w6_new_input)
w6_new_Y = np.atleast_1d(w6_new_output)

X_updated_w6 = np.vstack((X_updated_w5, w6_new_X))
Y_updated_w6 = np.append(Y_updated_w5, w6_new_Y)

X_updated_w6_scaled = scaler_x.fit_transform(X_updated_w6)
Y_updated_w6_scaled = scaler_y.fit_transform(Y_updated_w6.reshape(-1, 1))

df = pd.DataFrame(X_updated_w6, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w6

print(df.describe())

              x1         x2         x3         x4     target
count  36.000000  36.000000  36.000000  36.000000  36.000000
mean    0.526404   0.473235   0.448237   0.464039 -14.504143
std     0.270243   0.281606   0.241743   0.267265   8.984147
min     0.037825   0.006250   0.042186   0.081517 -32.625660
25%     0.315091   0.200495   0.267733   0.247716 -19.578543
50%     0.451380   0.487351   0.397840   0.428484 -14.152072
75%     0.740635   0.681607   0.692025   0.707228  -9.543919
max     0.985622   0.919592   0.939178   0.999483  -0.150467


In [16]:
model.fit(X_updated_w6_scaled, Y_updated_w6_scaled)

betas = [0.01, 0.10 , 0.25, 0.5, 1.0, 1.5, 2.5]
sweep_results = []

# Generate a Sobol sequence 
sampler = qmc.Sobol(d=4, scramble=True, seed=42)
sample_raw = sampler.random(n=8192)
sample_scaled = scaler_x.transform(sample_raw)

for b in betas:
    # Find the best starting point from the sample
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    # Use L-BFGS-B to find the absolute peak near that starting point
    bounds = [scaler_x.transform([[0,0,0,0]])[0], scaler_x.transform([[1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model, scaler_x),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)

    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })


# DISPLAY RESULTS
df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\n--- Optimized Model Parameters ---")
print(model.kernel_)

--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---
    Beta       x1       x2       x3       x4  Pred Mean  Uncertainty
0.010000 0.426289 0.400711 0.345209 0.414198  -0.358957     0.378135
0.100000 0.425827 0.399755 0.345675 0.414568  -0.359111     0.380911
0.250000 0.425009 0.398067 0.346579 0.415261  -0.360025     0.386082
0.500000 0.423512 0.394976 0.348493 0.416668  -0.363985     0.396503
1.000000 0.420127 0.387649 0.354263 0.420791  -0.387096     0.426531
1.500000 0.417190 0.378818 0.362927 0.427465  -0.445928     0.472989
2.500000 0.431229 0.356824 0.380997 0.451659  -0.749696     0.621792

--- Optimized Model Parameters ---
Matern(length_scale=[3.06, 2.54, 3.53, 2.87], nu=2.5) + WhiteKernel(noise_level=0.0011)


### Week 7 submission

Next point to query [0.425827 0.399755 0.345675 0.414568] beta = 0.1

* Found a new optima point - going into exploitation to try to improve this, give this another round of exploitation. 

In [17]:
# ---------------------------------------------------
# WEEK7: NEW DATA HERE
# ---------------------------------------------------

w7_new_input = 	[0.425827, 0.399755, 0.345675, 0.414568]
w7_new_output = 0.55211251515879

w7_new_X = np.atleast_2d(w7_new_input)
w7_new_Y = np.atleast_1d(w7_new_output)

X_updated_w7 = np.vstack((X_updated_w6, w7_new_X))
Y_updated_w7 = np.append(Y_updated_w6, w7_new_Y)

X_updated_w7_scaled = scaler_x.fit_transform(X_updated_w7)
Y_updated_w7_scaled = scaler_y.fit_transform(Y_updated_w7.reshape(-1, 1))

df = pd.DataFrame(X_updated_w7, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w7

print(df.describe())
# print(df.head(37))

              x1         x2         x3         x4     target
count  37.000000  37.000000  37.000000  37.000000  37.000000
mean    0.523686   0.471249   0.445465   0.462702 -14.097217
std     0.266976   0.277930   0.238958   0.263652   9.197804
min     0.037825   0.006250   0.042186   0.081517 -32.625660
25%     0.326076   0.211965   0.276520   0.249007 -19.441558
50%     0.448601   0.482103   0.384400   0.415644 -13.702747
75%     0.738613   0.666933   0.687152   0.701694  -7.966775
max     0.985622   0.919592   0.939178   0.999483   0.552113


In [18]:
model.fit(X_updated_w7_scaled, Y_updated_w7_scaled)

betas = [0.001, 0.01, 0.10 , 0.25, 0.5, 1.0]
sweep_results = []

# Generate a Sobol sequence 
sampler = qmc.Sobol(d=4, scramble=True, seed=42)
sample_raw = sampler.random(n=8192)
sample_scaled = scaler_x.transform(sample_raw)

for b in betas:
    # Find the best starting point from the sample
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    # Use L-BFGS-B to find the absolute peak near that starting point
    bounds = [scaler_x.transform([[0,0,0,0]])[0], scaler_x.transform([[1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model, scaler_x),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)

    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })


# DISPLAY RESULTS
df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\n--- Optimized Model Parameters ---")
print(model.kernel_)

--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---
    Beta       x1       x2       x3       x4  Pred Mean  Uncertainty
0.001000 0.416854 0.382890 0.353685 0.422870   0.008250     0.475682
0.010000 0.416807 0.382798 0.353755 0.422923   0.008248     0.476052
0.100000 0.416323 0.381865 0.354486 0.423483   0.008035     0.479901
0.250000 0.415484 0.380247 0.355839 0.424504   0.006791     0.486950
0.500000 0.413991 0.377367 0.358504 0.426485   0.001566     0.500730
1.000000 0.410716 0.370961 0.365651 0.431748  -0.026846     0.537828

--- Optimized Model Parameters ---
Matern(length_scale=[3.08, 2.51, 3.48, 2.86], nu=2.5) + WhiteKernel(noise_level=0.00159)


### Week 8 submission

Next point to query [0.416323 0.381865 0.354486 0.423483] beta = 0.1

* Very strange phenomenom, the first positive value in the series 
* Try extremely low beta to see if it's consistently positive number

In [19]:
# ---------------------------------------------------
# WEEK8: NEW DATA HERE
# ---------------------------------------------------

w8_new_input = 	[0.416323, 0.381865, 0.354486, 0.423483]
w8_new_output = 0.634425650963824

w8_new_X = np.atleast_2d(w8_new_input)
w8_new_Y = np.atleast_1d(w8_new_output)

X_updated_w8 = np.vstack((X_updated_w7, w8_new_X))
Y_updated_w8 = np.append(Y_updated_w7, w8_new_Y)

X_updated_w8_scaled = scaler_x.fit_transform(X_updated_w8)
Y_updated_w8_scaled = scaler_y.fit_transform(Y_updated_w8.reshape(-1, 1))

df = pd.DataFrame(X_updated_w8, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w8

# print(df.describe())
print(df.head(38))

          x1        x2        x3        x4     target
0   0.896981  0.725628  0.175404  0.701694 -22.108288
1   0.889356  0.499588  0.539269  0.508783 -14.601397
2   0.250946  0.033693  0.145380  0.494932 -11.699932
3   0.346962  0.006250  0.760564  0.613024 -16.053765
4   0.124871  0.129770  0.384400  0.287076 -10.069633
5   0.801303  0.500231  0.706645  0.195103 -15.487083
6   0.247708  0.060445  0.042186  0.441324 -12.681685
7   0.746702  0.757092  0.369353  0.206566 -16.026400
8   0.400665  0.072574  0.886768  0.243842 -17.049235
9   0.626071  0.586751  0.438806  0.778858 -12.741766
10  0.957135  0.597644  0.766114  0.776210 -27.316396
11  0.732812  0.145250  0.476813  0.133366 -13.527649
12  0.655115  0.072392  0.687152  0.081517 -16.679115
13  0.219734  0.832031  0.482864  0.082569 -16.507159
14  0.488594  0.211965  0.939178  0.376192 -17.817999
15  0.167130  0.876555  0.217240  0.959801 -26.561821
16  0.216911  0.166086  0.241372  0.770062 -12.758324
17  0.387488  0.804532  0.75

In [20]:
model.fit(X_updated_w8_scaled, Y_updated_w8_scaled)

betas = [0.001, 0.01, 0.10 , 0.25, 0.5, 1.0]
sweep_results = []

# Generate a Sobol sequence 
sampler = qmc.Sobol(d=4, scramble=True, seed=42)
sample_raw = sampler.random(n=8192)
sample_scaled = scaler_x.transform(sample_raw)

for b in betas:
    # Find the best starting point from the sample
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    # Use L-BFGS-B to find the absolute peak near that starting point
    bounds = [scaler_x.transform([[0,0,0,0]])[0], scaler_x.transform([[1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model, scaler_x),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)

    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })


# DISPLAY RESULTS
df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\n--- Optimized Model Parameters ---")
print(model.kernel_)

--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---
    Beta       x1       x2       x3       x4  Pred Mean  Uncertainty
0.001000 0.409197 0.367537 0.366975 0.431778   0.378786     0.444457
0.010000 0.409162 0.367465 0.367061 0.431840   0.378784     0.444822
0.100000 0.408815 0.366731 0.367943 0.432486   0.378573     0.448617
0.250000 0.408232 0.365466 0.369534 0.433659   0.377352     0.455536
0.500000 0.407261 0.363243 0.372549 0.435924   0.372254     0.468986
1.000000 0.405526 0.358355 0.380144 0.441971   0.344515     0.505193

--- Optimized Model Parameters ---
Matern(length_scale=[3.11, 2.5, 3.43, 2.86], nu=2.5) + WhiteKernel(noise_level=0.00125)


### Week 9 submission

Next point to query [0.408815 0.366731 0.367943 0.432486] beta = 0.1

* Second positive value in the series 
* Try again 
* New maximum point

In [22]:
# ---------------------------------------------------
# WEEK9: NEW DATA HERE
# ---------------------------------------------------

w9_new_input = [0.408815, 0.366731, 0.367943, 0.432486]
w9_new_output = 0.5652785895067969

w9_new_X = np.atleast_2d(w9_new_input)
w9_new_Y = np.atleast_1d(w9_new_output)

X_updated_w9 = np.vstack((X_updated_w8, w9_new_X))
Y_updated_w9 = np.append(Y_updated_w8, w9_new_Y)

X_updated_w9_scaled = scaler_x.fit_transform(X_updated_w9)
Y_updated_w9_scaled = scaler_y.fit_transform(Y_updated_w9.reshape(-1, 1))

df = pd.DataFrame(X_updated_w9, columns=['x1', 'x2', 'x3', 'x4'])
df['target'] = Y_updated_w9

# print(df.describe())
print(df.head(39))

          x1        x2        x3        x4     target
0   0.896981  0.725628  0.175404  0.701694 -22.108288
1   0.889356  0.499588  0.539269  0.508783 -14.601397
2   0.250946  0.033693  0.145380  0.494932 -11.699932
3   0.346962  0.006250  0.760564  0.613024 -16.053765
4   0.124871  0.129770  0.384400  0.287076 -10.069633
5   0.801303  0.500231  0.706645  0.195103 -15.487083
6   0.247708  0.060445  0.042186  0.441324 -12.681685
7   0.746702  0.757092  0.369353  0.206566 -16.026400
8   0.400665  0.072574  0.886768  0.243842 -17.049235
9   0.626071  0.586751  0.438806  0.778858 -12.741766
10  0.957135  0.597644  0.766114  0.776210 -27.316396
11  0.732812  0.145250  0.476813  0.133366 -13.527649
12  0.655115  0.072392  0.687152  0.081517 -16.679115
13  0.219734  0.832031  0.482864  0.082569 -16.507159
14  0.488594  0.211965  0.939178  0.376192 -17.817999
15  0.167130  0.876555  0.217240  0.959801 -26.561821
16  0.216911  0.166086  0.241372  0.770062 -12.758324
17  0.387488  0.804532  0.75

In [23]:
model.fit(X_updated_w9_scaled, Y_updated_w9_scaled)

betas = [0.001, 0.01, 0.10 , 0.25, 0.5, 1.0]
sweep_results = []

# Generate a Sobol sequence 
sampler = qmc.Sobol(d=4, scramble=True, seed=42)
sample_raw = sampler.random(n=8192)
sample_scaled = scaler_x.transform(sample_raw)

for b in betas:
    # Find the best starting point from the sample
    m, s = model.predict(sample_scaled, return_std=True)
    initial_ucb = m + (b * s)
    best_start_idx = np.argmax(initial_ucb)
    x0 = sample_scaled[best_start_idx]

    # Use L-BFGS-B to find the absolute peak near that starting point
    bounds = [scaler_x.transform([[0,0,0,0]])[0], scaler_x.transform([[1,1,1,1]])[0]]
    res = minimize(
        fun=acquisition_function,
        x0=x0,
        args=(b, model, scaler_x),
        method='L-BFGS-B',
        bounds=list(zip(bounds[0], bounds[1]))
    )
    
    best_x_scaled = res.x.reshape(1, -1)
    best_x_raw = scaler_x.inverse_transform(best_x_scaled)[0]
    
    final_mean_scaled, final_std_scaled = model.predict(best_x_scaled, return_std=True)

    actual_mean = scaler_y.inverse_transform([[final_mean_scaled[0]]])[0][0]
    actual_std = final_std_scaled[0] * scaler_y.scale_[0]
    
    sweep_results.append({
        'Beta': b,
        'x1': best_x_raw[0],
        'x2': best_x_raw[1],
        'x3': best_x_raw[2],
        'x4': best_x_raw[3],
        'Pred Mean': actual_mean,
        'Uncertainty': actual_std
    })


# DISPLAY RESULTS
df_sweep = pd.DataFrame(sweep_results)
print("--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---")
print(df_sweep.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
print("\n--- Optimized Model Parameters ---")
print(model.kernel_)

--- FUNCTION 4: NEXT QUERY SUGGESTIONS ---
    Beta       x1       x2       x3       x4  Pred Mean  Uncertainty
0.001000 0.407551 0.362838 0.372154 0.434821   0.497406     0.370158
0.010000 0.407534 0.362791 0.372217 0.434867   0.497405     0.370337
0.100000 0.407375 0.362309 0.372861 0.435340   0.497302     0.372191
0.250000 0.407134 0.361477 0.374021 0.436210   0.496700     0.375599
0.500000 0.406843 0.359988 0.376225 0.437937   0.494120     0.382392
1.000000 0.407803 0.356375 0.381859 0.443126   0.478228     0.402898

--- Optimized Model Parameters ---
Matern(length_scale=[3.17, 2.55, 3.47, 2.91], nu=2.5) + WhiteKernel(noise_level=0.000979)


### Week 10 submission

Next point to query [0.407134 0.361477 0.374021 0.436210] beta = 0.25

* Starting to accumulate positive points
* Try slightly more exploratory approach (beta = 0.25)
* Consider masking next run